# LeetCode 3833 — Count Dominant Indices bằng Python

Link đề: https://leetcode.com/problems/count-dominant-indices/ · Độ khó: **Easy** · Chủ đề: Array, Prefix/Suffix Sum, Enumeration

Skeleton LeetCode:

```python
class Solution:
    def dominantIndices(self, nums: List[int]) -> int:
        ...
```

> Ghi chú: tên bài, đề, ví dụ, constraints và chữ ký hàm đã được đối chiếu qua leetcode.doocs.org (Easy, không Premium).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu khái niệm **chỉ số trội (dominant index)**: `nums[i]` lớn **hơn hẳn** trung bình cộng của mọi phần tử **bên phải** nó.
2. Biết tính tổng đoạn bằng `sum(nums[i+1:])`, tổng hậu tố (suffix sum) và tổng chạy (running sum) từ phải sang trái.
3. Biết **tránh số thực**: so sánh `nums[i] * cnt > tổng` thay vì chia để so với trung bình.
4. Viết 5 cách (cắt lát + nhân chéo, `Fraction`, mảng tổng hậu tố, tổng chạy từ phải, tổng toàn bộ trừ tiền tố) và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh lỗi: tính cả phần tử cuối (chia cho 0), dùng `>=` thay `>`, `round` trung bình, quên cập nhật tổng.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` độ dài `n`. Chỉ số `i` được gọi là **trội** nếu `nums[i]` **lớn hơn hẳn** (strictly greater) **trung bình cộng** của các phần tử `nums[i+1], ..., nums[n-1]` (các phần tử bên phải). Phần tử ngoài cùng bên phải không có phần tử nào bên phải nên **không bao giờ** trội. Trả về **số lượng** chỉ số trội.

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[5,4,3]` | `2` | `i=0`: `5 > (4+3)/2 = 3.5`; `i=1`: `4 > 3`; `i=2` không có bên phải |
| 2 | `[4,1,2]` | `1` | `i=0`: `4 > 1.5` trội; `i=1`: `1 > 2` sai; `i=2` không trội |

Constraints: `1 <= nums.length <= 100`, `1 <= nums[i] <= 100`.

In [1]:
for nums in ([5, 4, 3], [4, 1, 2]):
    for i in range(len(nums) - 1):
        right = nums[i + 1:]
        print(f"nums={nums} i={i}: nums[i]={nums[i]} so với TB của {right} = {sum(right) / len(right):.2f}")

nums=[5, 4, 3] i=0: nums[i]=5 so với TB của [4, 3] = 3.50
nums=[5, 4, 3] i=1: nums[i]=4 so với TB của [3] = 3.00
nums=[4, 1, 2] i=0: nums[i]=4 so với TB của [1, 2] = 1.50
nums=[4, 1, 2] i=1: nums[i]=1 so với TB của [2] = 2.00


## 3. Kiến thức nền

### 3.1. Cắt lát (slicing) và `sum`

`nums[i+1:]` là mọi phần tử từ vị trí `i+1` đến cuối. Nếu `i+1` vượt biên, kết quả là list rỗng (không lỗi). `sum([])` bằng `0`.

In [2]:
nums = [5, 4, 3]
print(nums[1:], nums[2:], nums[3:])
print(sum(nums[1:]), sum(nums[3:]))

[4, 3] [3] []
7 0


### 3.2. Trung bình cộng và vì sao nên tránh chia

Trung bình của `k` số có tổng `S` là `S / k`. Phép chia trong Python 3 cho số thực (`float`), có thể gây sai số với số rất lớn. Ta so sánh `x > S / k` bằng cách nhân hai vế với `k > 0`: `x * k > S`. Hai điều kiện tương đương và chỉ dùng số nguyên.

In [3]:
x, S, k = 5, 7, 2           # 5 so với trung bình 3.5
print(x > S / k, x * k > S)
x, S, k = 3, 7, 2           # 3 so với 3.5
print(x > S / k, x * k > S)

True True
False False


### 3.3. `Fraction` — số hữu tỉ chính xác

Module `fractions` có `Fraction(a, b)` giữ đúng giá trị `a/b`, so sánh được với số nguyên mà không sai số.

In [4]:
from fractions import Fraction
avg = Fraction(7, 2)
print(avg, avg > 3, avg < 4, avg == 3.5)

7/2 True True True


### 3.4. Duyệt ngược và tổng chạy

`for i in range(n - 1, -1, -1)` duyệt từ phải sang trái. Giữ biến `run` là **tổng các phần tử đã đi qua** (phía bên phải `i`), sau mỗi bước cộng thêm `nums[i]`.

In [5]:
nums = [5, 4, 3]
run = 0
for i in range(len(nums) - 1, -1, -1):
    print(f"i={i} tổng bên phải={run}")
    run += nums[i]
print("tổng toàn mảng:", run)

i=2 tổng bên phải=0
i=1 tổng bên phải=3
i=0 tổng bên phải=7
tổng toàn mảng: 12


### 3.5. Tổng hậu tố bằng `itertools.accumulate`

`accumulate(reversed(nums))` trả các tổng tích lũy từ phải: với `[5,4,3]` ta có `3, 7, 12`. Đây là "suffix sum" tính một lần cho mọi vị trí.

In [6]:
from itertools import accumulate
nums = [5, 4, 3]
print(list(accumulate(reversed(nums))))
suffix = list(accumulate(reversed(nums)))[::-1]
print(suffix)           # suffix[i] = nums[i] + ... + nums[-1]

[3, 7, 12]
[12, 7, 3]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Với mỗi vị trí `i` (trừ vị trí cuối) ta cần hai thứ về "phần bên phải":

- **tổng** `S_i = nums[i+1] + ... + nums[n-1]`,
- **số phần tử** `k_i = n - 1 - i`.

`i` là trội khi `nums[i] > S_i / k_i`, tức `nums[i] * k_i > S_i`.

Tính tay `nums = [4, 1, 2]` (`n = 3`):

| i | nums[i] | phần bên phải | S_i | k_i | nums[i] × k_i | so với S_i | trội? |
|---|---|---|---|---|---|---|---|
| 0 | 4 | [1, 2] | 3 | 2 | 8 | 8 > 3 | có |
| 1 | 1 | [2] | 2 | 1 | 1 | 1 > 2 sai | không |
| 2 | 2 | [] | 0 | 0 | 0 | không xét | không |

Đếm: `1`. Còn với `[5, 4, 3]`: `i=0`: `5×2 = 10 > 7`; `i=1`: `4×1 = 4 > 3`; tổng 2.

In [7]:
def explain(nums):
    n = len(nums)
    count = 0
    for i in range(n - 1):
        S = sum(nums[i + 1:])
        k = n - 1 - i
        ok = nums[i] * k > S
        count += ok
        print(f"i={i}: {nums[i]}*{k}={nums[i] * k} vs S={S} -> {'trội' if ok else 'không'}")
    return count

print(explain([4, 1, 2]))
print(explain([5, 4, 3]))

i=0: 4*2=8 vs S=3 -> trội
i=1: 1*1=1 vs S=2 -> không
1
i=0: 5*2=10 vs S=7 -> trội
i=1: 4*1=4 vs S=3 -> trội
2


## 5. Cách 1 — Brute Force / baseline

Với mỗi `i` từ `0` đến `n - 2`: cắt `nums[i+1:]`, tính tổng bằng `sum`, độ dài bằng `len`, rồi so sánh bằng nhân chéo. Làm đúng theo đề.

Nút thắt: mỗi vị trí tính lại tổng đoạn phải từ đầu, `O(n)` mỗi lần, tổng `O(n²)`. Chạy nhanh với `n <= 100` nhưng lặp lại việc tính các tổng chồng lấn: đoạn phải của `i` là đoạn phải của `i+1` cộng thêm đúng một phần tử.

In [8]:
class SolutionBrute:
    def dominantIndices(self, nums):
        n = len(nums)
        count = 0
        for i in range(n - 1):                 # phần tử cuối không bao giờ trội
            right = nums[i + 1:]
            if nums[i] * len(right) > sum(right):
                count += 1
        return count

print(SolutionBrute().dominantIndices([5, 4, 3]), SolutionBrute().dominantIndices([4, 1, 2]))

2 1


**Độ phức tạp:** Time `O(n²)` (cắt lát và `sum` tốn `O(n)` mỗi vị trí), Space `O(n)` cho lát cắt.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** tổng bên phải của `i` và của `i-1` liên hệ nhau: `S_{i-1} = S_i + nums[i]`. Nên không cần tính lại: đi từ **phải sang trái** và giữ tổng chạy.

**Invariant:** khi xét vị trí `i`, biến `run` luôn bằng `nums[i+1] + ... + nums[n-1]` và `k = n - 1 - i` bằng số phần tử đó. Sau khi xét xong `i`, ta cộng `nums[i]` vào `run` (và `k` tăng 1) để chuẩn bị cho `i - 1`.

Quy tắc "nếu... thì...": nếu `k > 0` và `nums[i] * k > run` thì tăng đáp án; luôn cộng `nums[i]` vào `run` sau đó.

Các cách khác cùng ý: lưu sẵn mảng tổng hậu tố; hoặc lấy `total - prefix - nums[i]` đi từ trái sang phải.

## 7. Các cách tiếp theo

### Cách 2 — `Fraction` (so sánh chính xác bằng số hữu tỉ)

Giữ nguyên cấu trúc brute force nhưng tính trung bình bằng `Fraction(sum, len)` và so sánh trực tiếp `nums[i] > avg`. Dễ đọc nhất theo mô tả của đề, không dùng số thực nên không sai số.

In [9]:
from fractions import Fraction

class SolutionFraction:
    def dominantIndices(self, nums):
        count = 0
        for i in range(len(nums) - 1):
            right = nums[i + 1:]
            avg = Fraction(sum(right), len(right))
            if nums[i] > avg:
                count += 1
        return count

print(SolutionFraction().dominantIndices([5, 4, 3]), SolutionFraction().dominantIndices([4, 1, 2]))

2 1


**Độ phức tạp:** Time `O(n²)`, Space `O(n)`.

### Cách 3 — Mảng tổng hậu tố

Tính trước `suffix[i] = nums[i] + ... + nums[n-1]` (thêm `suffix[n] = 0`). Khi đó tổng bên phải `i` là `suffix[i+1]`, số phần tử `n - 1 - i`.

In [10]:
class SolutionSuffix:
    def dominantIndices(self, nums):
        n = len(nums)
        suffix = [0] * (n + 1)
        for i in range(n - 1, -1, -1):
            suffix[i] = suffix[i + 1] + nums[i]
        count = 0
        for i in range(n - 1):
            if nums[i] * (n - 1 - i) > suffix[i + 1]:
                count += 1
        return count

print(SolutionSuffix().dominantIndices([5, 4, 3]), SolutionSuffix().dominantIndices([4, 1, 2]))

2 1


**Độ phức tạp:** Time `O(n)`, Space `O(n)`.

### Cách 4 — Tổng chạy từ phải sang trái (tối ưu bộ nhớ)

Chỉ dùng một biến `run`, đúng như invariant ở mục 6.

In [11]:
class SolutionRunning:
    def dominantIndices(self, nums):
        count = 0
        run = 0                                  # tổng phần tử bên phải i
        k = 0                                    # số phần tử bên phải i
        for i in range(len(nums) - 1, -1, -1):
            if k > 0 and nums[i] * k > run:
                count += 1
            run += nums[i]
            k += 1
        return count

print(SolutionRunning().dominantIndices([5, 4, 3]), SolutionRunning().dominantIndices([4, 1, 2]))

2 1


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### Cách 5 — Tổng toàn bộ trừ tiền tố (đi từ trái sang phải)

Tính `total = sum(nums)`. Khi đứng ở `i`, gọi `left` là tổng các phần tử **trước** `i`; tổng bên phải là `total - left - nums[i]`. Cách này hữu ích khi dữ liệu đến theo thứ tự từ trái (không cần đảo chiều).

In [12]:
class SolutionPrefix:
    def dominantIndices(self, nums):
        n = len(nums)
        total = sum(nums)
        left = 0
        count = 0
        for i in range(n - 1):
            right_sum = total - left - nums[i]
            if nums[i] * (n - 1 - i) > right_sum:
                count += 1
            left += nums[i]
        return count

print(SolutionPrefix().dominantIndices([5, 4, 3]), SolutionPrefix().dominantIndices([4, 1, 2]))

2 1


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 4 trên `nums = [5, 4, 3]` (đi từ `i = 2` về `i = 0`):

| i | nums[i] | k (số phần tử phải) | run (tổng phải) | `nums[i]*k > run`? | count |
|---|---|---|---|---|---|
| 2 | 3 | 0 | 0 | k = 0, bỏ qua | 0 |
| 1 | 4 | 1 | 3 | 4 > 3: có | 1 |
| 0 | 5 | 2 | 7 | 10 > 7: có | 2 |

In [13]:
def trace_running(nums):
    count = 0
    run = 0
    k = 0
    for i in range(len(nums) - 1, -1, -1):
        if k > 0:
            ok = nums[i] * k > run
            print(f"i={i} nums[i]={nums[i]} k={k} run={run}: {nums[i] * k} > {run}? {ok}")
            count += ok
        else:
            print(f"i={i} nums[i]={nums[i]} k=0: phần tử cuối, bỏ qua")
        run += nums[i]
        k += 1
    print("kết quả:", count)
    return count

assert trace_running([5, 4, 3]) == 2
assert trace_running([4, 1, 2]) == 1

i=2 nums[i]=3 k=0: phần tử cuối, bỏ qua
i=1 nums[i]=4 k=1 run=3: 4 > 3? True
i=0 nums[i]=5 k=2 run=7: 10 > 7? True
kết quả: 2
i=2 nums[i]=2 k=0: phần tử cuối, bỏ qua
i=1 nums[i]=1 k=1 run=2: 1 > 2? False
i=0 nums[i]=4 k=2 run=3: 8 > 3? True
kết quả: 1


In [14]:
def trace_prefix(nums):
    n, total, left, count = len(nums), sum(nums), 0, 0
    for i in range(n - 1):
        right_sum = total - left - nums[i]
        k = n - 1 - i
        ok = nums[i] * k > right_sum
        print(f"i={i} left={left} right_sum={right_sum} k={k}: {nums[i]}*{k}={nums[i] * k} > {right_sum}? {ok}")
        count += ok
        left += nums[i]
    return count

assert trace_prefix([4, 1, 2]) == 1

i=0 left=0 right_sum=3 k=2: 4*2=8 > 3? True
i=1 left=4 right_sum=2 k=1: 1*1=1 > 2? False


## 9. Code nộp LeetCode

Dùng tổng chạy từ phải sang trái (Cách 4): `O(n)` thời gian, `O(1)` bộ nhớ.

In [15]:
from typing import List, Optional


class Solution:
    def dominantIndices(self, nums: List[int]) -> int:
        count = 0
        run = 0          # tổng các phần tử bên phải vị trí đang xét
        k = 0            # số phần tử bên phải vị trí đang xét
        for i in range(len(nums) - 1, -1, -1):
            if k > 0 and nums[i] * k > run:
                count += 1
            run += nums[i]
            k += 1
        return count


print(Solution().dominantIndices([5, 4, 3]), Solution().dominantIndices([4, 1, 2]))

2 1


## 10. Vì sao thuật toán đúng

1. **Tương đương số học.** Với `k > 0`, `nums[i] > S / k` tương đương `nums[i] * k > S` (nhân hai vế với số dương `k` giữ chiều bất đẳng thức). Mọi giá trị là số nguyên nên phép so sánh chính xác.
2. **Invariant của vòng lặp ngược.** Trước khi xử lý `i`: `run = sum(nums[i+1:])` và `k = n - 1 - i`. Ban đầu (`i = n - 1`) cả hai bằng `0` đúng. Sau khi xử lý ta cộng `nums[i]`, `k += 1` nên invariant còn đúng cho `i - 1`.
3. **Phần tử cuối.** Ở `i = n - 1` ta có `k = 0`, không có phần tử bên phải: theo đề không trội, đúng bằng nhánh `k > 0`.
4. **Đếm đủ.** Mỗi `i` được xét đúng một lần, nên `count` bằng đúng số chỉ số trội.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

- Cách 1, 2: mỗi `i` cắt lát và tính tổng `O(n)`, tổng `O(n²)`; phụ `O(n)` (lát cắt). Có `Fraction` thì hằng số lớn hơn.
- Cách 3: `O(n)` thời gian, `O(n)` mảng hậu tố.
- Cách 4, 5: `O(n)` thời gian, `O(1)` phụ.

Với `n <= 100` mọi cách đều nhanh; nhưng nếu đề cho `n = 10^5` thì chỉ các cách 3, 4, 5 qua được. Mọi cách đều không phụ thuộc phân phối dữ liệu (best = average = worst).

## 12. Bộ test

Đối chiếu cả 5 cách với ví dụ đề, biên (độ dài `1`, `2`, các số bằng nhau, tăng, giảm) và 300 mảng ngẫu nhiên đúng constraints.

In [16]:
import random

solutions = [SolutionBrute(), SolutionFraction(), SolutionSuffix(), SolutionRunning(), SolutionPrefix(), Solution()]

fixed = [
    ([5, 4, 3], 2),
    ([4, 1, 2], 1),
    ([1], 0),
    ([7, 7], 0),            # 7 không lớn hơn hẳn 7
    ([8, 7], 1),
    ([1, 2, 3, 4], 0),      # dãy tăng: không ai trội
    ([4, 3, 2, 1], 3),      # dãy giảm: mọi phần tử trừ cuối đều trội
    ([100] * 100, 0),
    ([2, 1, 2], 1),         # 2 > 1.5 trội, 1 > 2 sai
    ([3, 1, 5], 0),         # 3 không > 3
]
for nums, expected in fixed:
    for s in solutions:
        assert s.dominantIndices(list(nums)) == expected, (type(s).__name__, nums)

random.seed(38330)
for _ in range(300):
    n = random.randint(1, 100)
    hi = random.choice([2, 5, 100])
    nums = [random.randint(1, hi) for _ in range(n)]
    assert len({s.dominantIndices(list(nums)) for s in solutions}) == 1
print("tất cả test OK")

tất cả test OK


### Test thuộc tính

Phần tử cuối luôn không trội, nên đáp án tối đa là `n - 1`; thêm một phần tử rất lớn (100) vào **đầu** mảng thì chỉ số 0 trội trừ khi tất cả còn lại cũng bằng 100.

In [17]:
random.seed(3833)
for _ in range(200):
    n = random.randint(1, 60)
    nums = [random.randint(1, 100) for _ in range(n)]
    ans = Solution().dominantIndices(nums)
    assert 0 <= ans <= n - 1
    if n < 100:
        ext = [100] + nums
        res = Solution().dominantIndices(ext)
        first_dominant = 100 * n > sum(nums)
        # thêm vào đầu không ảnh hưởng các chỉ số phía sau nên đáp án mới chính là ans (+1 nếu chỉ số 0 trội)
        assert res == ans + int(first_dominant)
print("property test OK")

property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: xét cả phần tử cuối (chia cho 0)

Vòng `for i in range(n)` với `sum(nums[i+1:]) / len(nums[i+1:])` ở `i = n-1` chia cho `0` -> `ZeroDivisionError`.

In [18]:
def wrong_include_last(nums):
    count = 0
    for i in range(len(nums)):                 # SAI: phải range(n - 1)
        right = nums[i + 1:]
        if nums[i] > sum(right) / len(right):
            count += 1
    return count

try:
    wrong_include_last([5, 4, 3])
except ZeroDivisionError as e:
    print("ZeroDivisionError:", e)
assert Solution().dominantIndices([5, 4, 3]) == 2

ZeroDivisionError: division by zero


### Lỗi 2: dùng `>=` thay vì `>`

Đề yêu cầu **lớn hơn hẳn**. Với `[7, 7]`, `7 >= 7` là đúng nhưng `7 > 7` là sai.

In [19]:
def wrong_ge(nums):
    count = 0
    for i in range(len(nums) - 1):
        right = nums[i + 1:]
        if nums[i] * len(right) >= sum(right):      # SAI: phải là >
            count += 1
    return count

print(wrong_ge([7, 7]), "!=", Solution().dominantIndices([7, 7]))
assert wrong_ge([7, 7]) == 1 and Solution().dominantIndices([7, 7]) == 0

1 != 0


### Lỗi 3: làm tròn trung bình bằng `round`

`round(1.5)` là `2` nên `[2, 1, 2]` bị loại sai ở `i = 0` (thực tế `2 > 1.5`). `round` còn làm tròn "về số chẵn" (`round(2.5) = 2`) nên càng khó đoán.

In [20]:
def wrong_round(nums):
    count = 0
    for i in range(len(nums) - 1):
        right = nums[i + 1:]
        if nums[i] > round(sum(right) / len(right)):  # SAI: làm tròn trung bình
            count += 1
    return count

print(wrong_round([2, 1, 2]), "!=", Solution().dominantIndices([2, 1, 2]))
assert wrong_round([2, 1, 2]) == 0 and Solution().dominantIndices([2, 1, 2]) == 1

0 != 1


### Lỗi 4: so sánh với trung bình của **cả mảng** thay vì phần bên phải

Đề chỉ xét các phần tử **bên phải** `i`. Nếu dùng trung bình toàn mảng, ta tính cả phần tử bên trái và cả chính `nums[i]`.

In [21]:
def wrong_whole_mean(nums):
    mean = sum(nums) / len(nums)
    return sum(1 for i in range(len(nums) - 1) if nums[i] > mean)      # SAI: trung bình toàn mảng

print(wrong_whole_mean([1, 5, 5]), "!=", Solution().dominantIndices([1, 5, 5]))
assert wrong_whole_mean([1, 5, 5]) == 1 and Solution().dominantIndices([1, 5, 5]) == 0

1 != 0


### Lỗi 5: quên cộng `nums[i]` vào tổng chạy

Nếu trong vòng lặp ngược quên `run += nums[i]`, `run` luôn bằng `0` và mọi phần tử (trừ cuối) đều bị coi là trội.

In [22]:
def wrong_forget_run(nums):
    count = 0
    run, k = 0, 0
    for i in range(len(nums) - 1, -1, -1):
        if k > 0 and nums[i] * k > run:
            count += 1
        k += 1                                    # SAI: thiếu run += nums[i]
    return count

print(wrong_forget_run([4, 1, 2]), "!=", Solution().dominantIndices([4, 1, 2]))
assert wrong_forget_run([4, 1, 2]) == 2 and Solution().dominantIndices([4, 1, 2]) == 1

2 != 1


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "So sánh `nums[i]` với tổng/trung bình của một phía" -> nghĩ **prefix/suffix sum** hoặc tổng chạy.
- Tránh chia: `x > S / k` đổi thành `x * k > S` (khi `k > 0`).
- Khi mỗi vị trí cần "thông tin phía bên phải", hãy **duyệt ngược** và tích lũy.
- Kiểm tra biên: độ dài 1 (không có bên phải), các số bằng nhau (dấu `>` chặt), dãy tăng/giảm.
- Với `n <= 100`, brute force `O(n²)` luôn qua, nhưng nên luyện cách `O(n)` vì bài tương tự có thể có `n` lớn.

## 15. Biến thể của bài

1. **Leaders in an Array** (GFG) / "Right Leaders": phần tử lớn hơn **mọi** phần tử bên phải (so với max, thay vì trung bình).
2. **Product of Array Except Self** (238): tích tiền tố/hậu tố.
3. **Find Pivot Index** (724): tổng trái bằng tổng phải.
4. **Count of Smaller Numbers After Self** (315): bản khó hơn, đếm số phần tử nhỏ hơn bên phải.
5. **Average of Levels / Maximum Average Subarray I** (643): làm việc với trung bình đoạn.
6. Biến thể: trội nếu `nums[i]` **lớn hơn hoặc bằng** trung bình, hoặc so sánh với trung bình bên **trái**.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | cắt lát, `sum`, nhân chéo | `O(n²)` | `O(n)` | Oracle, `n` nhỏ |
| 2. Fraction | trung bình chính xác | `O(n²)` | `O(n)` | Đọc sát đề, kiểm tra |
| 3. Suffix array | tổng hậu tố tính sẵn | `O(n)` | `O(n)` | Khi cần truy vấn nhiều lần |
| 4. Tổng chạy | duyệt ngược, 1 biến | `O(n)` | `O(1)` | Nộp bài |
| 5. total - prefix | duyệt xuôi | `O(n)` | `O(1)` | Dữ liệu đến theo thứ tự trái -> phải |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().dominantIndices([6, 3, 3])` và `Solution().dominantIndices([1, 1, 1])` ra gì?

**Bài 2 (tính tay):** với `nums = [9, 1, 8, 2]`, lập bảng `S_i`, `k_i` và chỉ ra chỉ số nào trội.

**Bài 3 (điền TODO):** hoàn thành `count_greater_than_right_max(nums)`: đếm chỉ số `i` mà `nums[i]` lớn hơn hẳn **mọi** phần tử bên phải (chỉ số cuối tính là trội theo định nghĩa "mọi phần tử của tập rỗng").

**Bài 4 (tự viết lại):** viết `dominant_indices_list(nums)` trả về **danh sách chỉ số** trội thay vì số lượng.

**Bài 5 (mini challenge):** viết `count_dominant_left(nums)`: đếm `i` sao cho `nums[i]` lớn hơn hẳn trung bình các phần tử **bên trái** (`i = 0` không trội).

In [23]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().dominantIndices([6, 3, 3]), Solution().dominantIndices([1, 1, 1]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def count_greater_than_right_max_todo(nums):
    count = 0
    # TODO: duyệt ngược, giữ max bên phải (bắt đầu bằng 0 vì nums[i] >= 1)
    return count

1 0


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: `[6,3,3]` -> `i=0`: `6 > 3`; `i=1`: `3 > 3`? Bài 2: `S_0 = 11`, `k_0 = 3`.

**Mức 2 (vừa):** Bài 3: duyệt ngược, giữ `best = 0`; nếu `nums[i] > best` thì đếm, sau đó `best = max(best, nums[i])`. Bài 4: giống Cách 4 nhưng `append(i)` rồi đảo kết quả. Bài 5: dùng tổng chạy từ trái và `k = i`.

**Mức 3 (gần đáp án):** Bài 1: `1` và `0`. Bài 2: `i=0`: `9*3=27 > 11` trội; `i=1`: `1*2 > 10` sai; `i=2`: `8 > 2` trội -> chỉ số `[0, 2]`. Bài 5: kết quả với `[1, 3, 2]` là `1`.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2
assert Solution().dominantIndices([6, 3, 3]) == 1 and Solution().dominantIndices([1, 1, 1]) == 0
assert Solution().dominantIndices([9, 1, 8, 2]) == 2

# Bài 3
def count_greater_than_right_max(nums):
    count, best = 0, 0
    for x in reversed(nums):
        if x > best:
            count += 1
        best = max(best, x)
    return count
assert count_greater_than_right_max([5, 4, 3]) == 3
assert count_greater_than_right_max([4, 4, 4]) == 1 and count_greater_than_right_max([1, 2, 3]) == 1

# Bài 4
def dominant_indices_list(nums):
    res, run, k = [], 0, 0
    for i in range(len(nums) - 1, -1, -1):
        if k > 0 and nums[i] * k > run:
            res.append(i)
        run += nums[i]
        k += 1
    return res[::-1]
assert dominant_indices_list([9, 1, 8, 2]) == [0, 2]
assert dominant_indices_list([5, 4, 3]) == [0, 1] and dominant_indices_list([1]) == []

# Bài 5
def count_dominant_left(nums):
    count, run = 0, 0
    for i, x in enumerate(nums):
        if i > 0 and x * i > run:
            count += 1
        run += x
    return count
assert count_dominant_left([1, 3, 2]) == 1
assert count_dominant_left([1, 1, 1]) == 0 and count_dominant_left([1, 5, 5]) == 2
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "so sánh phần tử với tổng/trung bình của phía bên phải":**

```python
count = 0
run = 0                                  # tổng bên phải
k = 0                                    # số phần tử bên phải
for i in range(len(nums) - 1, -1, -1):
    if k > 0 and nums[i] * k > run:      # nhân chéo, không chia
        count += 1
    run += nums[i]
    k += 1
return count
```

**Checklist trước khi submit:**

- [ ] Đã bỏ qua phần tử cuối (không có phần tử bên phải)?
- [ ] Dùng `>` chặt (lớn hơn **hẳn**), không `>=`?
- [ ] Không làm tròn trung bình; dùng nhân chéo hoặc `Fraction`?
- [ ] Tổng bên phải **không** chứa `nums[i]`?
- [ ] Đã cập nhật `run` và `k` ở mỗi vòng?
- [ ] Đã thử độ dài 1, các số bằng nhau, dãy tăng và dãy giảm?
- [ ] Chữ ký đúng `dominantIndices(self, nums: List[int]) -> int`?